# Tutorial 0: Audio Generation Quickstart

<p align="right" style="margin-right: 8px;">
    <a target="_blank" href="https://colab.research.google.com/github/idiap/sdialog/blob/main/tutorials/audio/0.quickstart.ipynb">
        <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
    </a>
</p>

`dialog.to_audio()` turns any two-speaker dialogue into realistic audio and returns it **together with its ground-truth annotations**: who speaks when, what is said, and word timestamps. The same text dialogues can produce audio at different levels of difficulty, for ASR, speaker diarization, speech LLMs, call-center analytics and other speech tasks.

In this tutorial you will learn how to:
- generate audio from a dialogue and access its annotations,
- choose voices (matched to the personas, designed, random or your own),
- use scene presets (clean, room, call center, ...) in mono or stereo,
- control timing (pauses, overlaps) and acoustics (reverb, noise, telephone channel),
- generate audio for many dialogues in parallel.

> The default TTS engine is [Dia2](https://github.com/nari-labs/dia2), which needs Python >= 3.10 and works best on a GPU (on Colab, select a GPU runtime).

## Environment Setup

In [1]:
# Setup the environment depending on whether we are running in Google Colab or Jupyter Notebook
import os
from IPython import get_ipython

if "google.colab" in str(get_ipython()):
    print("Running on CoLab")
    # Installing sdialog
    !git clone https://github.com/idiap/sdialog.git
    %cd sdialog
    %pip install -e .[audio]
    %cd ..
else:
    print("Running in Jupyter Notebook")

Running in Jupyter Notebook


### Load the example dialogue

We start from an existing doctor-patient dialogue (generated as in the previous tutorials). Its personas include gender and age, which are used to choose matching voices.

In [2]:
from sdialog import Dialog

path_dialog = "../../tests/data/demo_dialog_doctor_patient.json"

if not os.path.exists(path_dialog):
    !wget https://raw.githubusercontent.com/idiap/sdialog/refs/heads/main/tests/data/demo_dialog_doctor_patient.json
    path_dialog = "./demo_dialog_doctor_patient.json"

dialog = Dialog.from_file(path_dialog)
dialog.print()

[dialog_id] 86ff0e9e-4b2e-4379-87d9-8c6000424b4f
[model] {'name': 'amazon:anthropic.claude-3-5-sonnet-20240620-v1:0', 'temperature': 0.7, 'max_tokens': 512, 'region_name': 'us-east-1'}
[seed] 42
--- Dialogue Begins ---
[Marie] Hello doctor, I've been having headaches for two weeks and I'm very tired.
[Dr. Martin] I'm sorry to hear that, Marie. Can you tell me where the pain is strongest?
[Marie] Mostly at the front of my head and behind my eyes.
[Dr. Martin] I see. Have you been under a lot of stress lately?
[Marie] Yes, work has been stressful and my sleep hasn't been great.
[Dr. Martin] That could explain the tension. I'd like to check your blood pressure now.
[Marie] Of course, go ahead.
[Dr. Martin] Your blood pressure looks normal, but your neck muscles are very tense.
[Marie] So it might just be stress?
[Dr. Martin] Most likely. I recommend stretching, better sleep habits, and keeping a headache diary.
[Marie] Thank you, doctor. I'll start doing that.
--- Dialogue Ends ---


## 1. From text to audio in one line

The first call loads Dia2 (and downloads its weights the first time), so it takes a bit longer. Passing a `seed` makes the result reproducible.

In [3]:
audio = dialog.to_audio(seed=0)

print(f"{audio.duration:.1f} seconds, {audio.sample_rate} Hz, {audio.channels} channel(s)")
audio.display()

Importing the dtw module. When using in academic works please cite:
  T. Giorgino. Computing and Visualizing Dynamic Time Warping Alignments in R: The dtw Package.
  J. Stat. Soft., doi:10.18637/jss.v031.i07.



Detected language: English


Detected language: English


35.6 seconds, 16000 Hz, 1 channel(s)


### Annotations

`audio.segments` has one segment per turn, with its speaker, text, start/end times and word timestamps:

In [4]:
for seg in audio.segments:
    print(f"{seg.start:6.2f} - {seg.end:6.2f}  {seg.speaker:>10}: {seg.text}")

print(audio.segments[0].words[:5])  # (word, start time) pairs

  0.30 -   4.91       Marie: Hello doctor, I've been having headaches for two weeks and I'm very tired.
  4.91 -   8.35  Dr. Martin: I'm sorry to hear that, Marie. Can you tell me where the pain is strongest?
  8.35 -  10.91       Marie: Mostly at the front of my head and behind my eyes.
 10.91 -  13.23  Dr. Martin: I see. Have you been under a lot of stress lately?
 13.28 -  16.99       Marie: Yes, work has been stressful and my sleep hasn't been great.
 16.99 -  20.59  Dr. Martin: That could explain the tension. I'd like to check your blood pressure now.
 20.59 -  22.27       Marie: Of course, go ahead.
 22.27 -  25.55  Dr. Martin: Your blood pressure looks normal, but your neck muscles are very tense.
 25.55 -  27.31       Marie: So it might just be stress?
 27.31 -  32.19  Dr. Martin: Most likely. I recommend stretching, better sleep habits, and keeping a headache diary.
 32.64 -  35.06       Marie: Thank you, doctor. I'll start doing that.
[('doctor,', np.float64(0.51)), ("I've", 

The annotations can be exported as RTTM (speaker diarization) and JSON, and `save()` writes the audio file plus both annotation files:

In [5]:
print(audio.to_rttm())

paths = audio.save("output/quickstart/clean")
paths

SPEAKER 86ff0e9e-4b2e-4379-87d9-8c6000424b4f 1 0.300 4.610 <NA> <NA> Marie <NA> <NA>
SPEAKER 86ff0e9e-4b2e-4379-87d9-8c6000424b4f 1 4.910 3.440 <NA> <NA> Dr._Martin <NA> <NA>
SPEAKER 86ff0e9e-4b2e-4379-87d9-8c6000424b4f 1 8.350 2.560 <NA> <NA> Marie <NA> <NA>
SPEAKER 86ff0e9e-4b2e-4379-87d9-8c6000424b4f 1 10.910 2.320 <NA> <NA> Dr._Martin <NA> <NA>
SPEAKER 86ff0e9e-4b2e-4379-87d9-8c6000424b4f 1 13.280 3.710 <NA> <NA> Marie <NA> <NA>
SPEAKER 86ff0e9e-4b2e-4379-87d9-8c6000424b4f 1 16.990 3.600 <NA> <NA> Dr._Martin <NA> <NA>
SPEAKER 86ff0e9e-4b2e-4379-87d9-8c6000424b4f 1 20.590 1.680 <NA> <NA> Marie <NA> <NA>
SPEAKER 86ff0e9e-4b2e-4379-87d9-8c6000424b4f 1 22.270 3.280 <NA> <NA> Dr._Martin <NA> <NA>
SPEAKER 86ff0e9e-4b2e-4379-87d9-8c6000424b4f 1 25.550 1.760 <NA> <NA> Marie <NA> <NA>
SPEAKER 86ff0e9e-4b2e-4379-87d9-8c6000424b4f 1 27.310 4.880 <NA> <NA> Dr._Martin <NA> <NA>
SPEAKER 86ff0e9e-4b2e-4379-87d9-8c6000424b4f 1 32.640 2.420 <NA> <NA> Marie <NA> <NA>



{'audio': 'output/quickstart/clean.wav',
 'json': 'output/quickstart/clean.json',
 'rttm': 'output/quickstart/clean.rttm'}

`audio.voices` tells which voice each speaker got, and `audio.config` stores every parameter that was used, including the values drawn at random. Together with the seed, this is enough to reproduce the audio exactly.

In [6]:
print(audio.voices)
print(audio.config)

{'Marie': {'source': 'db', 'id': '6914', 'gender': 'female', 'age': -1, 'language': 'english'}, 'Dr. Martin': {'source': 'db', 'id': '1746', 'gender': 'male', 'age': -1, 'language': 'english'}}
{'seed': 0, 'preset': None, 'scene': {'pauses': 'natural', 'overlap': 0.0, 'max_overlap': 0.3, 'reverb': None, 'noise': None, 'channel': None, 'level_jitter_db': 0.0}, 'stereo': False, 'sample_rate': 16000, 'tts': {'engine': 'Dia2TTS', 'model': 'nari-labs/Dia2-2B', 'cfg_scale': 2.0, 'text_temperature': 0.6, 'audio_temperature': 0.8, 'top_k': 50, 'max_window_words': 170}, 'level_offsets_db': {'Marie': -0.0, 'Dr. Martin': -0.0}, 'room': None, 'noise': None, 'channel': None}


## 2. Voices

- `voices="db"` (default): voices from a voice database (`sdialog/voices-libritts`) that match each persona's gender, age and language.
- `voices="random"`: voices made up by Dia2, giving new voices for every seed.
- `voices="design"`: voices designed from each persona's description with Qwen3-TTS VoiceDesign (`pip install qwen-tts`).
- A dictionary `{speaker: voice}` with your own reference audio files (or `(array, sample_rate)` tuples).

In [7]:
for seed in (1, 2):
    random_voices = dialog.to_audio(voices="random", seed=seed)
    random_voices.display()

In [8]:
# Use your own reference voices, e.g. the ones selected in the first example
# my_voices = dialog.to_audio(voices={"Dr. Martin": "voices/doctor.wav", "Marie": "voices/patient.wav"})

# Voices designed from the personas (requires `pip install qwen-tts`)
# designed = dialog.to_audio(voices="design", voice_description="{{ gender }}, {{ age }} years old, warm and calm voice.")

## 3. Presets: from clean to hard

Presets set realistic values for timing and acoustics:

| preset | scene |
|---|---|
| `clean` | natural TTS timing, no effects (default) |
| `conversational` | shorter pauses, overlapping turns |
| `room` | both speakers in a medium room, light noise |
| `call_center` | agent on a headset, caller through a phone line with noise |
| `far_field` | distant microphone in a large reverberant room, noisy |
| `hard` | large room, very noisy, telephone channel, frequent overlaps |

Voices are chosen first from the seed, so using the same seed keeps the same voices across presets, which makes them easy to compare:

In [9]:
from sdialog.audio import list_presets

for preset in list_presets():
    print(f"\n=== {preset} ===")
    preset_audio = dialog.to_audio(preset=preset, seed=0)
    print(preset_audio.config["room"], preset_audio.config["noise"], preset_audio.config["channel"])
    preset_audio.display()


=== clean ===


None None None



=== conversational ===


None None None



=== room ===


{'type': 'simulated', 'size': 'medium', 'dimensions': [7.99, 5.96, 3.04], 'rt60': 0.595, 'microphone': [5.31, 2.43, 0.89], 'sources': [[2.9, 2.04, 1.39], [6.83, 1.17, 1.76]]} {'all': {'type': 'pink', 'snr_db': 23.58}} None


... (2 outputs left out)


=== far_field ===


{'type': 'simulated', 'size': 'large', 'dimensions': [19.97, 14.87, 5.21], 'rt60': 1.025, 'microphone': [13.56, 5.89, 0.89], 'sources': [[8.74, 5.12, 1.39], [16.59, 3.37, 1.76]]} {'all': {'type': 'pink', 'snr_db': 8.58}} None



=== hard ===


{'type': 'simulated', 'size': 'large', 'dimensions': [19.81, 12.8, 5.13], 'rt60': 1.044, 'microphone': [7.81, 2.09, 1.31], 'sources': [[6.3, 5.91, 1.49], [12.9, 0.5, 1.41]]} {'all': {'type': 'pink', 'snr_db': 5.72}} {'all': 'g711'}


### Mono or stereo

A preset only sets the scene, so any preset can be rendered in mono or in stereo (one channel per speaker, in order of appearance). For instance, call-center recordings usually store the agent and the caller on separate channels:

In [10]:
stereo = dialog.to_audio(preset="call_center", stereo=True, seed=0)
mono = dialog.to_audio(preset="call_center", stereo=False, seed=0)

print(stereo.audio.shape, mono.audio.shape)
print([(seg.speaker, seg.channel) for seg in stereo.segments[:4]])
stereo.display()

(619089, 2) (635409,)
[('Marie', 0), ('Dr. Martin', 1), ('Marie', 0), ('Dr. Martin', 1)]


## 4. Full control

Every preset value can be overridden, or set without a preset:

- `pauses`: `"natural"`, a median in seconds, or a `(min, max)` range
- `overlap` / `max_overlap`: probability of overlapping turns / maximum overlap (fraction of the previous turn)
- `reverb`: `"small"`, `"medium"`, `"large"` (simulated room of random size), a `Room` of your choice (see the [next tutorial](1.rooms_and_acoustics.ipynb)), or a path to measured impulse responses
- `noise`: SNR in dB, `(min, max)` SNR range, or `{"type": "white" | "pink" | "brown" | <audio path>, "snr": ...}`
- `channel`: telephone codec, `"g711"`, `"g729"`, `"amr"` or `"amr-wb"`
- `level_jitter_db`: loudness difference between speakers

`reverb`, `noise` and `channel` take one value for the whole recording, or a dictionary with one value per speaker:

In [11]:
custom = dialog.to_audio(
    reverb="large",                                        # same large room for both speakers
    noise={"Marie": {"type": "pink", "snr": (5, 10)}},     # noise only on the patient's side
    channel={"Marie": "amr"},                              # the patient calls through a phone line
    pauses=(0.1, 0.4),
    overlap=0.3,
    seed=0,
)
custom.display()

In [12]:
# Start from a preset and remove one of its effects with False
hard_no_phone = dialog.to_audio(preset="hard", channel=False, seed=0)
hard_no_phone.display()

You can also register your own presets:

In [13]:
from sdialog.audio import register_preset

register_preset("noisy_cafe_call", channel="g711", noise={"type": "brown", "snr": (0, 10)},
                pauses=0.3, overlap=0.25, level_jitter_db=4)

dialog.to_audio(preset="noisy_cafe_call", seed=0).display()

## 5. Many dialogues in parallel

`dialogs_to_audio` converts a list of dialogues with one TTS model per worker. By default it uses every available GPU, or the CPU if there is none; `devices` can also be `"cpu"`, `"cuda:1"` or a list of devices. It saves the audio and annotations of each dialogue, plus an `index.jsonl` file listing all of them:

In [14]:
from sdialog.audio import dialogs_to_audio

dialogs = [dialog.clone() for _ in range(3)]  # e.g. a list of generated dialogues (each with its own id)

rows = dialogs_to_audio(dialogs, "output/quickstart/batch", preset="room")
rows

Detected language: English


Detected language: English


Detected language: English


Detected language: English


Detected language: English


Detected language: English


[{'index': 0,
  'dialog_id': 'b9353c16-e538-4f89-9019-b7c52d349b31',
  'audio': 'output/quickstart/batch/b9353c16-e538-4f89-9019-b7c52d349b31.wav',
  'json': 'output/quickstart/batch/b9353c16-e538-4f89-9019-b7c52d349b31.json',
  'rttm': 'output/quickstart/batch/b9353c16-e538-4f89-9019-b7c52d349b31.rttm',
  'duration': 44.05,
  'seed': 1289236349},
 {'index': 1,
  'dialog_id': '54c25744-ddcd-45de-b424-0b63ef3a38b7',
  'audio': 'output/quickstart/batch/54c25744-ddcd-45de-b424-0b63ef3a38b7.wav',
  'json': 'output/quickstart/batch/54c25744-ddcd-45de-b424-0b63ef3a38b7.json',
  'rttm': 'output/quickstart/batch/54c25744-ddcd-45de-b424-0b63ef3a38b7.rttm',
  'duration': 39.896,
  'seed': 1857210015},
 {'index': 2,
  'dialog_id': '6dcf00a2-c5af-4f6b-b7bc-c2ffe68af9ff',
  'audio': 'output/quickstart/batch/6dcf00a2-c5af-4f6b-b7bc-c2ffe68af9ff.wav',
  'json': 'output/quickstart/batch/6dcf00a2-c5af-4f6b-b7bc-c2ffe68af9ff.json',
  'rttm': 'output/quickstart/batch/6dcf00a2-c5af-4f6b-b7bc-c2ffe68af9ff.

## 6. Other TTS engines

Dia2 generates the whole conversation at once, with natural turn-taking. Classic one-utterance-at-a-time engines, such as Kokoro (`pip install kokoro`) or Qwen3-TTS, can be used too; they also work on Python 3.9:

```python
from sdialog.audio.tts import KokoroTTS

audio = dialog.to_audio(tts=KokoroTTS(), preset="room")
```

The [next tutorial](1.rooms_and_acoustics.ipynb) shows how to choose the room your dialogues are recorded in: its size and materials, the microphone and speaker positions, directional microphones, and a different room for each speaker.